## Feature Engineering and Model Preparation

Notebook goal:

Prepare the data for ML model by: 
- Selecting modeling features
- Engineering additional predictors 
- Create train/test/val sets

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [ ]:


df = pd.read_parquet("dataset/processed/open_food_facts_cleaned.parquet")

print("Dataset shape:", df.shape)
df.head(3)

Dataset shape: (83074, 20)


,code,product_name,brands,categories_en,countries_en,pnns_groups_1,pnns_groups_2,nutrition_grade_fr,nova_group,additives_n,additives_tags,ingredients_text,energy_100g,fat_100g,saturated_fat_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g
0,1247,Shake Mix Vanilla Flavour,Herbalife,Dietary supplements,"India,Ireland",unknown,unknown,<NA>,4,6.0,"en:e304,en:e304i,en:e340,en:e340ii,en:e341,en:...","Soy Protein Isolate, Sugar, Fructose Powder, W...",1688.400000,7.200000,1.6,42.000000,29.600000,12.0,36.000000,1.230000
1,127534563,German fine bread,None,"Plant-based foods and beverages,Plant-based fo...",United States,Cereals and potatoes,Bread,c,3,0.0,None,"Rye flour, water, whole wheat flour, bran flou...",1199.920000,3.510000,0.0,52.630000,8.770000,7.0,7.020000,1.710000
2,1328,Harvest whole wheat bread,"Trader Joe's, E.Leclerc","Plant-based foods and beverages,Plant-based fo...",United States,Sugary snacks,Pastries,c,4,1.0,"en:e322,en:e322i","WHOLE WHEAT FLOUR, WATER, WHEAT GLUTEN, HONEY,...",1081.395349,4.651163,0.0,41.860465,4.651163,2.0,11.627907,1.046512


## Feature Selection

We select the key numerical features that represent nutritional composition and processing indicators. These features will be used as inputs to the machine learning models, while `nova_group` is used as the target variable.

In [2]:
# define our target
TARGET = "nova_group"

# define our numeric features
NUMERIC_FEATURES = [
    "energy_100g",
    "fat_100g",
    "carbohydrates_100g",
    "sugars_100g",
    "fiber_100g",
    "proteins_100g",
    "salt_100g",
    "saturated_fat_100g",
    "additives_n"
]

# create our modeling dataframe
df_model = df[NUMERIC_FEATURES + [TARGET]].copy()

print("Model dataset shape:", df_model.shape)
df_model.head(3)

Model dataset shape: (83074, 10)


,energy_100g,fat_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,saturated_fat_100g,additives_n,nova_group
0,1688.400000,7.200000,42.000000,29.600000,12.0,36.000000,1.230000,1.6,6.0,4
1,1199.920000,3.510000,52.630000,8.770000,7.0,7.020000,1.710000,0.0,0.0,3
2,1081.395349,4.651163,41.860465,4.651163,2.0,11.627907,1.046512,0.0,1.0,4


## Feature Engineering

To improve our model performance, we create the additional features below. 
- sugar_fiber_ratio
- fat_protein_ratio
- additives_per_energy

In [3]:
# avoid division by zero, in case any of our values are zero
# from the nutrional label. A small epsilon will take care of this
epsilon = 1e-5

# sugar to fiber ratio (processed foods tend to be high sugar, low fiber)
df_model["sugar_fiber_ratio"] = df_model["sugars_100g"] / (df_model["fiber_100g"] + epsilon)

# fat to protein ratio (helps distinguish nutrient balance)
# again, processed foods tend of have higher fat relative to protein content 
df_model["fat_protein_ratio"] = df_model["fat_100g"] / (df_model["proteins_100g"] + epsilon)

# additive density relative to calories
# higher value can indicate many additives for the caloric count
df_model["additives_per_energy"] = df_model["additives_n"] / (df_model["energy_100g"] + 1)

print("Updated dataset shape:", df_model.shape)
df_model.head(3)

Updated dataset shape: (83074, 13)


,energy_100g,fat_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,saturated_fat_100g,additives_n,nova_group,sugar_fiber_ratio,fat_protein_ratio,additives_per_energy
0,1688.400000,7.200000,42.000000,29.600000,12.0,36.000000,1.230000,1.6,6.0,4,2.466665,0.200000,0.003552
1,1199.920000,3.510000,52.630000,8.770000,7.0,7.020000,1.710000,0.0,0.0,3,1.252855,0.499999,0.000000
2,1081.395349,4.651163,41.860465,4.651163,2.0,11.627907,1.046512,0.0,1.0,4,2.325570,0.400000,0.000924


## Train/Test/Validation Split

In [ ]:

# Split features and target
X = df_model.drop(columns=["nova_group"])
y = df_model["nova_group"]

# Step 1: Train (60%) + Temp (40%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.4,
    stratify=y,
    random_state=42
)

# Step 2: Split Temp into Validation (20%) and Test (20%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

Train shape: (49844, 12)
Validation shape: (16615, 12)
Test shape: (16615, 12)


## Feature Scaling

- Feature sacling is applied after splitting our data, so that statistics from the training set do not leak into the validation or test sets. 
- Standardization is required because we will use a Deep Learning model to perform anomaly detection of the processed foods. 

In [7]:
# initialize scaler
scaler = StandardScaler()

# fit on training data only
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Scaled train shape:", X_train_scaled.shape)
print("Scaled validation shape:", X_val_scaled.shape)
print("Scaled test shape:", X_test_scaled.shape)

Scaled train shape: (49844, 12)
Scaled validation shape: (16615, 12)
Scaled test shape: (16615, 12)
